# Qwen 프롬프트 랩 — RunPod 라이브 + 긴 컨텍스트 검증

RunPod Serverless 로 서빙 중인 Qwen 에 대고 **노드별 프롬프트를 그 자리에서 고치고**,
**긴 컨텍스트에서 출력이 어떻게 망가지는지**(예: 입력에 없던 "토익" 같은 환각 태그)를
한 노트북에서 확인한다. system/user 포맷·파서는 모두 런타임 코드를 그대로 import 해
**train==serve** 를 보장한다.

대상 노드: `splitter`(단일 TODO, adapter=base) · `judge` · `plan_generator`(adapter=planner).

**필요 환경변수**: `RUNPOD_PLANNER_ENDPOINT_URL`, `RUNPOD_API_KEY`.
실행 전에 셸에서 export 하거나 아래 Setup 셀에서 직접 채운다.

> Jupyter 는 top-level `await` 를 지원하므로 셀에서 바로 `await complete(...)` 해도 된다.

## 1. Setup — repo root · 런타임 import · 엔드포인트

In [ ]:
import os, sys, json, asyncio, pathlib
from datetime import date

# repo root(mongle-ai) 를 sys.path 에. 노트북은 sft_pipeline/eval/ 에 있다.
ROOT = pathlib.Path.cwd()
while not (ROOT / "adapters" / "todo_creation" / "_prompts.py").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("mongle-ai repo root 를 못 찾음 — 노트북을 repo 안에서 실행하세요")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

# 런타임 전송(공용 RunPod 클라이언트)
from adapters._shared.runpod_client import run_and_poll, RunPodJobError

# 런타임 프롬프트(편집 출발점) — 복붙 금지, import 로 train==serve 유지
from adapters.todo_creation._prompts import (
    TASK_SPLITTER_SYSTEM, task_splitter_user,
    PLANNER_JUDGE_SYSTEM, planner_judge_user,
    PLAN_GENERATOR_SYSTEM, plan_generator_user,
    GOAL_TAG_SYSTEM, goal_tag_user,
)
# 런타임 파서 — 출력이 라이브에서 실제로 어떻게 파싱되는지 그대로 검증
from adapters.todo_creation.qwen_llm import (
    parse_task_response, _parse_json_object, _parse_plan_days, strip_json_fence,
)

# repo .env 자동 로드(이미 셸에 export 된 값이 우선) — Jupyter 는 .env 를 안 읽으므로
_envfile = ROOT / ".env"
if _envfile.exists():
    for line in _envfile.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip())

# 엔드포인트/키 — env(.env 포함) 우선, 없으면 여기서 직접 채운다
ENDPOINT_URL = os.environ.get("RUNPOD_PLANNER_ENDPOINT_URL", "").strip()
API_KEY = os.environ.get("RUNPOD_API_KEY", "").strip() or "EMPTY"
# ENDPOINT_URL = "https://api.runpod.ai/v2/<your-endpoint-id>"   # ← 필요하면 직접
# API_KEY = "<your-runpod-api-key>"
assert ENDPOINT_URL, "RUNPOD_PLANNER_ENDPOINT_URL 을 env 나 위 줄에서 설정하세요"
TODAY = date(2026, 6, 23)
print("endpoint:", ENDPOINT_URL, "| key set:", API_KEY != "EMPTY")

## 2. 호출 헬퍼 — `complete()` 하나로 모든 노드

In [ ]:
async def complete(messages, *, adapter="base", max_tokens=800, temperature=0.1, poll_timeout=300.0):
    """RunPod Serverless 호출 → 생성 텍스트(output.text). 런타임 payload 계약과 동일."""
    payload = {"input": {"adapter": adapter, "messages": messages,
                         "temperature": temperature, "max_tokens": max_tokens}}
    try:
        out = await run_and_poll(endpoint_url=ENDPOINT_URL, api_key=API_KEY,
                                 payload=payload, label="lab", poll_timeout=poll_timeout)
    except RunPodJobError as err:
        return f"<<RunPodJobError: {err}>>"
    text = out.get("text")
    return str(text) if text is not None else f"<<output.text 없음: {out!r}>>"

def ctx_size(messages):
    """대략적 컨텍스트 크기: 메시지 수 + 총 문자수(+가능하면 토큰수)."""
    chars = sum(len(m["content"]) for m in messages)
    info = f"{len(messages)} msgs / {chars} chars"
    try:  # ponytail: tokenizer 있으면 정확히, 없으면 char 만. 둘 다 충분
        from transformers import AutoTokenizer
        global _TOK
        _TOK = globals().get("_TOK") or AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
        toks = _TOK.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
        info += f" / {len(toks)} tokens"
    except Exception:
        info += " / ~{} tokens(추정)".format(chars // 2)
    return info

print("complete() / ctx_size() 준비 완료")

## 3. 긴 컨텍스트 빌더 + 환각 누수 탐지

In [ ]:
# 긴 컨텍스트를 만드는 두 가지 축:
#  - splitter: user prompt 자체를 길게 (단일턴)
#  - judge/plan_generator: history 를 여러 턴으로 (멀티턴)

_FILLER_TURNS = [
    ("user", "오늘 날씨 어때? 그냥 궁금해서."),
    ("assistant", "날씨는 제가 모르지만, 할 일 정리는 도와드릴 수 있어요!"),
    ("user", "어제 마트에서 우유랑 계란 샀어. 별 얘기 아님."),
    ("assistant", "메모해둘게요. 더 필요한 거 있으면 말씀해주세요."),
    ("user", "회사에서 회식했는데 늦게 끝났어."),
    ("assistant", "고생하셨어요. 일정에 반영할 게 있을까요?"),
]

def make_history(n_turns):
    """n_turns 만큼 잡담 히스토리를 채운다(멀티턴 노드의 긴 컨텍스트 재현용)."""
    turns = []
    while len(turns) < n_turns:
        role, content = _FILLER_TURNS[len(turns) % len(_FILLER_TURNS)]
        turns.append({"role": role, "content": content})
    return turns[:n_turns]

def pad_prompt(prompt, n_repeat):
    """splitter용: 무의미 문장을 덧붙여 prompt 를 길게(단일턴 긴 컨텍스트)."""
    noise = " 그리고 별로 안 중요한데 그냥 적어두는 잡담이야."
    return prompt + noise * n_repeat

# 입력에 등장하지 않은 시험/주제명이 출력에 새면 환각 누수
LEAK_WATCH = ["토익", "TOEIC", "정처기", "정보처리기사", "한국사", "SQLD", "컴활", "토플", "오픽", "OPIC"]

def find_leaks(text, user_input):
    """user_input 에 없는데 출력 text 에 나타난 watchlist 용어 = 누수 의심."""
    return [w for w in LEAK_WATCH if w in text and w not in user_input]

print("make_history / pad_prompt / find_leaks 준비 완료")

## 4. 노드별 편집 + 테스트

각 노드 셀은 **`*_SYSTEM` 변수에 런타임 프롬프트를 그대로 담아 시작**한다.
고치고 싶으면 그 줄을 삼중따옴표 문자열로 덮어쓴 뒤 셀을 다시 실행하면 된다.
바로 위 `print(...)` 셀로 런타임 원문을 출력해 복사 출발점으로 쓴다.

### 4-① splitter (단일 TODO · adapter=base) — 긴 prompt 누수 재현

In [ ]:
print(TASK_SPLITTER_SYSTEM)   # ← 편집 출발점: 이 내용을 복사해 아래 SPLITTER_SYSTEM 을 덮어쓰면 됨

In [ ]:
# 편집 지점: 기본은 런타임 프롬프트 그대로. 고치려면 아래를 삼중따옴표로 덮어쓰세요.
SPLITTER_SYSTEM = TASK_SPLITTER_SYSTEM
# SPLITTER_SYSTEM = """
# 너는 ... (여기에 수정본)
# """

PROMPT = "내일 기출 풀고 모레 오답 정리할래"
N_REPEAT = 0          # 0=짧게 / 30,80,150... 으로 올려 긴 컨텍스트 누수 관찰
prompt = pad_prompt(PROMPT, N_REPEAT)

msgs = [{"role": "system", "content": SPLITTER_SYSTEM},
        {"role": "user", "content": task_splitter_user(prompt)}]
print("context:", ctx_size(msgs))
raw = await complete(msgs, adapter="base", max_tokens=400)
print("\n--- raw ---\n", raw)
try:
    parsed = parse_task_response(raw, TODAY)
    print("\n--- parsed ---\n", parsed.model_dump_json(indent=2))
    print("PARSE OK · intent =", parsed.intent, "· tasks =", len(parsed.tasks))
except Exception as e:
    print("\nPARSE FAIL:", e)
leaks = find_leaks(raw, PROMPT)
print("누수 의심 용어:", leaks or "없음")

### 4-② judge (멀티턴 · adapter=planner) — 긴 history 누수 재현

In [ ]:
print(PLANNER_JUDGE_SYSTEM)

In [ ]:
JUDGE_SYSTEM = PLANNER_JUDGE_SYSTEM
# JUDGE_SYSTEM = """
# 너는 ... (수정본)
# """

MESSAGE = "다음 주 금요일에 발표 준비 끝내야 해. 하루 2시간 정도 가능."
N_TURNS = 0           # 0=짧게 / 6,12,24... 로 올려 멀티턴 긴 컨텍스트 관찰
history = make_history(N_TURNS)

user = planner_judge_user(history=history, message=MESSAGE, today=TODAY, user_profile_memory=None)
msgs = [{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": user}]
print("context:", ctx_size(msgs))
raw = await complete(msgs, adapter="planner", max_tokens=500)
print("\n--- raw ---\n", raw)
try:
    parsed = _parse_json_object(raw)
    print("\n--- parsed ---\n", json.dumps(parsed, ensure_ascii=False, indent=2))
    pg = parsed.get("parsed_goal", {})
    print("PARSE OK · intent =", parsed.get("intent"), "· goal_tag =", pg.get("goal_tag"))
except Exception as e:
    print("\nPARSE FAIL:", e)
print("누수 의심 용어:", find_leaks(raw, MESSAGE) or "없음")

### 4-③ plan_generator (멀티턴 · adapter=planner) — days-grid 출력

In [ ]:
print(PLAN_GENERATOR_SYSTEM)

In [ ]:
PLANGEN_SYSTEM = PLAN_GENERATOR_SYSTEM
# PLANGEN_SYSTEM = """
# 너는 ... (수정본)
# """

# plan_generator 입력은 judge 가 만든 parsed_goal. 직접 채워 테스트.
parsed_goal = {
    "intent": "plan",
    "goal_text": "다음 주 금요일 발표 준비",
    "goal_tag": "발표준비",
    "deadline": "2026-06-30",
    "daily_capacity_minutes": 120,
    "personalization_patch": {"preferences": [], "constraints": []},
}
user = plan_generator_user(parsed_goal=parsed_goal, today=TODAY)
msgs = [{"role": "system", "content": PLANGEN_SYSTEM}, {"role": "user", "content": user}]
print("context:", ctx_size(msgs))
raw = await complete(msgs, adapter="planner", max_tokens=900)
print("\n--- raw ---\n", raw)
try:
    parsed = _parse_json_object(raw)
    days = _parse_plan_days(parsed.get("days"))
    # 계약 불변식 점검(런타임은 관대하지만, 좋은 출력은 지켜야 함)
    n_tasks = sum(len(d["tasks"]) for d in days)
    bad_due = [(d["date"], t.due_date) for d in days for t in d["tasks"] if t.due_date != d["date"]]
    print("\nPARSE OK · days =", len(days), "· tasks =", n_tasks)
    print("summary:", (parsed.get("summary_text") or "")[:120])
    print("계약 위반(due_date!=day.date):", bad_due or "없음",
          "| >7일:", len(days) > 7, "| >12 task:", n_tasks > 12)
except Exception as e:
    print("\nPARSE FAIL:", e)
print("누수 의심 용어:", find_leaks(raw, parsed_goal["goal_text"]) or "없음")

## 5. 런타임 프롬프트 vs 편집 프롬프트 — 같은 긴 입력으로 A/B

splitter 예시. 위 4-① 에서 `SPLITTER_SYSTEM` 을 고친 뒤 이 셀을 돌리면
**런타임 원본 vs 내 편집본**을 같은 긴 컨텍스트 입력에 대해 나란히 비교한다.

In [ ]:
AB_PROMPT = "내일 기출 풀고 모레 오답 정리할래"
AB_REPEAT = 80   # 누수가 잘 나는 길이로

prompt = pad_prompt(AB_PROMPT, AB_REPEAT)
for label, system in [("BASELINE(런타임)", TASK_SPLITTER_SYSTEM), ("EDITED(내 편집본)", SPLITTER_SYSTEM)]:
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": task_splitter_user(prompt)}]
    raw = await complete(msgs, adapter="base", max_tokens=400)
    try:
        parsed = parse_task_response(raw, TODAY)
        summary = f"intent={parsed.intent} tasks={[t.title for t in parsed.tasks]}"
        ok = "OK"
    except Exception as e:
        summary, ok = str(e), "FAIL"
    leaks = find_leaks(raw, AB_PROMPT)
    print(f"[{label}] {ctx_size(msgs)}  parse={ok}  누수={leaks or '없음'}")
    print("   ", summary, "\n")